# Phonological Adaptor Grammar Segmenter
### Bayesian word segmentation with a MaxEnt UR→SR channel and HMC weight inference

**Architecture overview**

| Layer | What it does |
|---|---|
| Pitman-Yor cache | Emergent lexicon of underlying representations (URs) with power-law frequency prior |
| MaxEnt phonology | Maps each UR to a probability distribution over surface realisations via learnable OT-style constraint weights |
| HMC sampler | Draws full posterior samples over constraint weights — no point estimates |
| Gibbs segmenter | Samples word boundary positions and UR assignments, conditioned on current weight sample |

**Inference loop (Option B — interleaved blocks)**

```
for K Gibbs sweeps:
    for each utterance:
        sample new segmentation + URs   (conditioned on current w)
        accumulate (UR, SR) pairs
run L HMC leapfrog steps → draw new w from p(w | (UR, SR) pairs, prior)
repeat
```

**References**  
- Breiss & Wilson (2020). *Extending adaptor grammars to learn phonological alternations.* SCiL.  
- Goldwater, Griffiths & Johnson (2009). *A Bayesian framework for word segmentation.* Cognition.  
- Hayes & Wilson (2008). *A maximum entropy model of phonotactics.* LI.  
- Johnson et al. (2006). *Adaptor grammars.* NeurIPS.  
- Neal (2011). *MCMC using Hamiltonian dynamics.* Handbook of MCMC.  
- Pakman & Paninski (2014). *Exact Hamiltonian Monte Carlo for truncated multivariate Gaussians.* JCGS.  
- Steriade (2001). *Directional asymmetries in place assimilation.* (P-map.)  
- Zuraw (2013). \*MAP constraints. UCLA ms.  


## 1. Imports and optional WFST backend

`pynini` compiles the MaxEnt grammar to a log-semiring WFST, giving an exact
partition function Z(UR, **w**) via `compose + shortestdistance`.  
If not installed, the code falls back to a DP over a bounded candidate set (len(UR) ± 2).

Install with: `conda install -c conda-forge pynini`


In [1]:
from __future__ import annotations
import math
import random
import warnings
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

import numpy as np

# ── optional pynini import ────────────────────────────────────────────────
try:
    import pynini
    from pynini.lib import byte
    PYNINI_AVAILABLE = True
    print("pynini found — using WFST backend for exact Z(UR, w)")
except ImportError:
    PYNINI_AVAILABLE = False
    warnings.warn(
        "pynini not found – falling back to pure-Python DP for channel "
        "probabilities.  Install with: conda install -c conda-forge pynini",
        ImportWarning, stacklevel=2)
    print("pynini not found — using DP approximation (len(UR) ± 2 candidates)")


pynini found — using WFST backend for exact Z(UR, w)


## 2. Phonological feature tables and p-map distance

The **p-map** (Steriade 2001) says that phonological alternations between
perceptually similar sounds are less costly than those between dissimilar sounds.
We encode this as a simple manner/voicing feature table.  
`pmap_distance(c1, c2)` returns a cost in [0, 4]:

| Relationship | Example | Cost |
|---|---|---|
| Identity | t → t | 0.0 |
| Same manner, same voicing | t → d | 0.5 |
| Same manner, diff voicing | p → b | 1.0 |
| Cross-manner | t → s | 2.0+ |

This initialises the `FAITH-PMAP` constraint weight: cheap alternations get a 
lower initial penalty and can be "learned down" more easily by the data.


In [2]:
# Simplified manner classes for p-map initialisation (Steriade 2001)
MANNER = {
    'p': 'stop',  'b': 'stop',  't': 'stop',  'd': 'stop',
    'k': 'stop',  'g': 'stop',
    'f': 'fric',  'v': 'fric',  's': 'fric',  'z': 'fric',
    'S': 'fric',  'Z': 'fric',  'h': 'fric',
    'm': 'nasal', 'n': 'nasal', 'N': 'nasal',
    'l': 'liquid','r': 'liquid',
    'w': 'glide', 'j': 'glide',
    'i': 'vowel', 'I': 'vowel', 'e': 'vowel', 'E': 'vowel',
    'a': 'vowel', 'A': 'vowel', 'o': 'vowel', 'O': 'vowel',
    'u': 'vowel', 'U': 'vowel', '@': 'vowel',
}

VOICING = {
    'p': '-', 'b': '+', 't': '-', 'd': '+', 'k': '-', 'g': '+',
    'f': '-', 'v': '+', 's': '-', 'z': '+', 'S': '-', 'Z': '+',
    'm': '+', 'n': '+', 'N': '+', 'l': '+', 'r': '+',
    'w': '+', 'j': '+',
    'i': '+', 'I': '+', 'e': '+', 'E': '+', 'a': '+', 'A': '+',
    'o': '+', 'O': '+', 'u': '+', 'U': '+', '@': '+',
}


def pmap_distance(c1: str, c2: str) -> float:
    """
    Rough perceptual distance between two phoneme symbols.
    Returns a value in [0, 4]; higher = more distinct = higher faithfulness cost.
    Based on Steriade (2001): changes within a class cost less than across classes.
    """
    if c1 == c2:
        return 0.0
    m1 = MANNER.get(c1, 'other')
    m2 = MANNER.get(c2, 'other')
    v1 = VOICING.get(c1, '?')
    v2 = VOICING.get(c2, '?')
    cost = 0.0
    if m1 != m2:
        cost += 2.0   # cross-manner change is expensive
    else:
        cost += 0.5   # within-manner (e.g. voicing alternation)
    if v1 != v2:
        cost += 0.5
    return cost


# Quick sanity check
print(f"pmap_distance('t', 't') = {pmap_distance('t', 't')}   (identity → 0)")
print(f"pmap_distance('t', 'd') = {pmap_distance('t', 'd')}   (same manner, voicing change → 0.5+0.5=1.0)")
print(f"pmap_distance('d', 'D') = {pmap_distance('d', 'D')}   (cross-manner → 2.0)")  # D not in table
print(f"pmap_distance('t', 's') = {pmap_distance('t', 's')}   (stop→fric cross-manner → 2.5)")


pmap_distance('t', 't') = 0.0   (identity → 0)
pmap_distance('t', 'd') = 1.0   (same manner, voicing change → 0.5+0.5=1.0)
pmap_distance('d', 'D') = 2.5   (cross-manner → 2.0)
pmap_distance('t', 's') = 2.0   (stop→fric cross-manner → 2.5)


## 3. MaxEnt constraint classes

Three OT-style constraints, each implementing `violations(ur, sr) → float`:

| Constraint | Penalises | Init weight |
|---|---|---|
| `FAITH-PMAP` | Any UR→SR change, scaled by p-map perceptual distance | High (p-map prior) |
| `MAX` | Deletion of a UR segment with no SR correspondent | Low |
| `DEP` | Epenthesis — SR segment with no UR correspondent | Low |

The constraint set is extensible: subclass `Constraint` and pass a custom list
to `MaxEntPhonology(constraints=[...])`.  Context-sensitive markedness constraints
(e.g. *CODA-OBSTRUENT, positional faithfulness) can be added this way.


In [3]:
class Constraint:
    """Abstract base for a single (markedness or faithfulness) constraint."""
    name: str

    def violations(self, ur: str, sr: str) -> float:
        """Return the violation count V(ur, sr) for this constraint."""
        raise NotImplementedError


class PmapFaithConstraint(Constraint):
    """
    FAITH-PMAP: graded faithfulness constraint whose per-pair cost equals
    pmap_distance(ur_char, sr_char) summed over all aligned positions.
    Captures the p-map insight: perceptually cheap changes cost less (Zuraw 2013).
    """
    def __init__(self):
        self.name = 'FAITH-PMAP'

    def violations(self, ur: str, sr: str) -> float:
        alignment = _levenshtein_alignment(ur, sr)
        v = 0.0
        for (u_c, s_c) in alignment:
            if u_c is None:
                v += 1.0   # epenthesis
            elif s_c is None:
                v += 1.0   # deletion
            else:
                v += pmap_distance(u_c, s_c)
        return v


class MaxVConstraint(Constraint):
    """MAX: one violation per UR segment deleted (no SR correspondent)."""
    def __init__(self):
        self.name = 'MAX'

    def violations(self, ur: str, sr: str) -> float:
        alignment = _levenshtein_alignment(ur, sr)
        return sum(1.0 for (u_c, s_c) in alignment if u_c is not None and s_c is None)


class DepConstraint(Constraint):
    """DEP: one violation per SR segment inserted (no UR correspondent)."""
    def __init__(self):
        self.name = 'DEP'

    def violations(self, ur: str, sr: str) -> float:
        alignment = _levenshtein_alignment(ur, sr)
        return sum(1.0 for (u_c, s_c) in alignment if u_c is None)


def _default_constraint_set() -> List[Constraint]:
    """
    Default constraint set: FAITH-PMAP, MAX, DEP.
    Extend by subclassing Constraint and passing a custom list to MaxEntPhonology.
    """
    return [
        PmapFaithConstraint(),
        MaxVConstraint(),
        DepConstraint(),
    ]


## 4. MaxEnt phonological grammar with HMC weight inference

### The channel model

$$P(\text{SR} \mid \text{UR},\, \mathbf{w}) = \frac{\exp\!\left(-\sum_c w_c \cdot V_c(\text{UR},\text{SR})\right)}{Z(\text{UR},\mathbf{w})}$$

where $V_c$ is the violation count of constraint $c$, and $Z$ is the partition
function summing over all candidate SRs within len(UR) ± 2 characters.

### HMC for weight posteriors

Rather than finding a point-estimate for **w**, we sample from the posterior:

$$p(\mathbf{w} \mid \{(u_i, s_i)\}) \propto \exp\!\left(\sum_i \log P(s_i \mid u_i, \mathbf{w})\right) \cdot p_{\text{prior}}(\mathbf{w})$$

The prior is a **half-normal** centred at the p-map initial value:  
$p_{\text{prior}}(w_c) \propto \exp(-w_c^2 / 2\sigma_c^2)$ for $w_c \geq 0$,  
where $\sigma_c = $ initial weight.

HMC uses leapfrog integration with a **reflection trick** at the $w=0$ boundary
(Pakman & Paninski 2014): when a leapfrog step would push a weight negative,
both the position and the corresponding momentum are negated. This is
volume-preserving and maintains detailed balance — no Jacobian correction needed.

> **Note on cell ordering:** `MaxEntPhonology` internally calls `_logsumexp`, `_levenshtein_alignment`, and `_random_edit`, which are defined in §9 (Utilities). This is intentional — Python resolves these names at call-time. Run all cells top-to-bottom and everything will work correctly.


In [4]:
class MaxEntPhonology:
    """
    MaxEnt phonological grammar with Bayesian HMC inference over constraint weights.

    Key methods:
        harmony(ur, sr)          → float        H = Σ_c w_c · V_c(ur, sr)
        log_prob(ur, sr)         → float        log P(SR | UR, w)
        accumulate(ur, sr)                      record a (UR, SR) observation
        run_hmc(n_samples, ...)                 draw posterior weight samples
        lexicon_ur(ur_counts)    → {ur: sr}     MAP SR for each UR

    Performance caches (all weight-independent unless noted):
        _viol_cache   {(ur,sr) → np.ndarray}   violation vectors — never invalidated
        _lp_cache     {(ur,sr) → float}         log P(SR|UR,w)   — cleared on weight update
        _cand_cache   {ur → frozenset}           candidate SR sets — never invalidated
    """

    def __init__(self,
                 alphabet: str,
                 constraints: Optional[List[Constraint]] = None,
                 pmap_init_weight: float = 5.0,
                 hmc_step_size: float = 0.05,
                 hmc_leapfrog_steps: int = 10):
        self.alphabet = list(alphabet)
        self.constraints = constraints or _default_constraint_set()
        self.n_constraints = len(self.constraints)

        # Initialise weights: FAITH-PMAP high (p-map prior), others near 0
        self.weights = np.zeros(self.n_constraints)
        for i, c in enumerate(self.constraints):
            self.weights[i] = pmap_init_weight if ('PMAP' in c.name or 'FAITH' in c.name) else 0.1

        self._initial_weights = self.weights.copy()   # snapshot for display

        # HMC state
        self.weight_history: List[np.ndarray] = []    # full posterior chain
        self.hmc_step_size: float = hmc_step_size     # adapted during warm-up
        self.hmc_leapfrog_steps: int = hmc_leapfrog_steps
        self._hmc_accept_history: List[float] = []    # per-block accept rates

        # Training data buffer (flushed after each HMC block)
        self._train_pairs: List[Tuple[str, str]] = []

        # ── Performance caches ──────────────────────────────────────────────
        # Violation vectors are weight-independent: cache permanently.
        self._viol_cache: Dict[Tuple[str,str], np.ndarray] = {}
        # Candidate SR sets are UR-dependent only: cache permanently.
        self._cand_cache: Dict[str, frozenset] = {}
        # log P(SR|UR,w) depends on weights: cleared whenever weights change.
        self._lp_cache: Dict[Tuple[str,str], float] = {}

        # WFST cache (kept for completeness but not used in hot path)
        self._wfst: Optional[object] = None
        self._wfst_dirty = True

    # ── violation scoring ─────────────────────────────────────────────────

    def _violation_vector(self, ur: str, sr: str) -> np.ndarray:
        """Return cached violation vector V(ur, sr) — shape [n_constraints].
        Weight-independent, so this cache is never invalidated."""
        key = (ur, sr)
        if key not in self._viol_cache:
            self._viol_cache[key] = np.array(
                [c.violations(ur, sr) for c in self.constraints], dtype=float)
        return self._viol_cache[key]

    def _candidates_for(self, ur: str, n_random: int = 50) -> frozenset:
        """Return cached set of candidate SRs for a given UR (len ± 2 bound).
        UR-dependent only, so this cache is never invalidated."""
        if ur not in self._cand_cache:
            min_len = max(1, len(ur) - 2)
            max_len = len(ur) + 2
            cands: set = {ur}
            for i in range(len(ur)):
                for c in self.alphabet:
                    cand = ur[:i] + c + ur[i+1:]
                    if min_len <= len(cand) <= max_len:
                        cands.add(cand)
                cand = ur[:i] + ur[i+1:]
                if min_len <= len(cand) <= max_len:
                    cands.add(cand)
            for i in range(len(ur) + 1):
                for c in self.alphabet:
                    cand = ur[:i] + c + ur[i:]
                    if min_len <= len(cand) <= max_len:
                        cands.add(cand)
            for _ in range(n_random):
                cand = _random_edit(_random_edit(ur, self.alphabet), self.alphabet)
                if min_len <= len(cand) <= max_len:
                    cands.add(cand)
            self._cand_cache[ur] = frozenset(cands)
        return self._cand_cache[ur]

    def harmony(self, ur: str, sr: str) -> float:
        """H(ur, sr) = Σ_c  w_c · V_c(ur, sr).  Lower harmony = better candidate."""
        return float(self.weights @ self._violation_vector(ur, sr))

    def log_prob(self, ur: str, sr: str) -> float:
        """log P(SR | UR, w).
        Result is cached keyed by (ur, sr); cache is cleared whenever weights change."""
        key = (ur, sr)
        if key not in self._lp_cache:
            self._lp_cache[key] = self._log_prob_dp(ur, sr)
        return self._lp_cache[key]

    # ── pure-Python DP ────────────────────────────────────────────────────

    def _log_prob_dp(self, ur: str, sr: str) -> float:
        """
        log P(SR | UR, w) via bounded candidate enumeration.
        Uses cached violation vectors and candidate sets — only the
        final dot products with current weights are computed fresh.
        """
        candidates = self._candidates_for(ur) | {sr}
        # Harmony scores: dot product of current weights with cached viol vectors
        h_sr     = float(self.weights @ self._violation_vector(ur, sr))
        h_cands  = np.array([float(self.weights @ self._violation_vector(ur, c))
                             for c in candidates])
        log_denom = _logsumexp_np(-h_cands)
        return -h_sr - log_denom

    # ── pynini WFST (kept for reference; not used in hot path) ───────────

    def _compile_wfst(self) -> None:
        """Compile MaxEnt grammar to a log-semiring WFST (pynini required)."""
        if not PYNINI_AVAILABLE:
            return
        w_pmap = float(self.weights[next(i for i,c in enumerate(self.constraints) if 'PMAP' in c.name)])
        w_max  = float(self.weights[next(i for i,c in enumerate(self.constraints) if c.name == 'MAX')])
        w_dep  = float(self.weights[next(i for i,c in enumerate(self.constraints) if c.name == 'DEP')])

        pairs = []
        for u in self.alphabet:
            for s in self.alphabet:
                pairs.append((u, s, w_pmap * pmap_distance(u, s)))
        for u in self.alphabet:
            pairs.append((u, '', w_max))
        for s in self.alphabet:
            pairs.append(('', s, w_dep))

        single_op = None
        for (u_str, s_str, cost) in pairs:
            weight = pynini.Weight('log', str(cost))
            arc = pynini.cross(
                pynini.accep(u_str, arc_type='log') if u_str else pynini.Fst(arc_type='log'),
                pynini.accep(s_str, arc_type='log') if s_str else pynini.Fst(arc_type='log'),
            )
            arc = pynini.arcmap(arc, map_type='times', weight=weight)
            single_op = pynini.union(single_op, arc) if single_op else arc
        self._wfst = pynini.closure(single_op)
        self._wfst_dirty = False

    def _log_prob_wfst(self, ur: str, sr: str) -> float:
        """log P(SR | UR) via WFST compose+intersect (exact Z). Not used in hot path."""
        if self._wfst_dirty:
            self._compile_wfst()
        try:
            composed = pynini.compose(pynini.accep(ur, arc_type='log'), self._wfst)
            projected = pynini.project(composed, 'output')
            intersected = pynini.intersect(projected, pynini.accep(sr, arc_type='log'))
            d = pynini.shortestdistance(intersected, reverse=True)
            return -float(d[0]) if len(d) > 0 else -1e30
        except Exception:
            return self._log_prob_dp(ur, sr)

    # ── HMC weight inference ──────────────────────────────────────────────

    def accumulate(self, ur: str, sr: str) -> None:
        """Record a (UR, SR) pair for the next HMC block."""
        self._train_pairs.append((ur, sr))

    def _build_pair_data(self) -> List[Tuple[np.ndarray, np.ndarray]]:
        """Precompute violation matrices for the current training batch.
        Uses cached violation vectors — O(1) per pair after first call."""
        pair_data = []
        for (ur, sr) in self._train_pairs:
            v_obs   = self._violation_vector(ur, sr)
            cands   = self._candidates_for(ur) | {sr}
            v_cands = np.array([self._violation_vector(ur, cand) for cand in cands])
            pair_data.append((v_obs, v_cands))
        return pair_data

    def _log_posterior_and_grad(
            self,
            w: np.ndarray,
            pair_data: List[Tuple[np.ndarray, np.ndarray]],
    ) -> Tuple[float, np.ndarray]:
        """
        Potential energy U = -log p(w | data) and gradient ∇_w U.

        Likelihood:  Σ_{(u,s)} [ H(u,s,w) + log Z(u,w) ]
        Prior:       Σ_c  w_c² / (2 σ_c²)   (half-normal, σ_c = initial weight)

        Returns (U, grad_U) — both positive (HMC minimises U).
        """
        n_c = len(w)
        U = 0.0
        grad_U = np.zeros(n_c)

        for (v_obs, v_cands) in pair_data:
            h_obs   = w @ v_obs
            h_cands = v_cands @ w
            log_z   = _logsumexp_np(-h_cands)
            U      += h_obs + log_z
            softmax = np.exp(-h_cands - log_z)
            E_v     = softmax @ v_cands
            grad_U += v_obs - E_v

        sigma2  = np.maximum(self._initial_weights ** 2, 0.01)
        U      += 0.5 * np.sum(w ** 2 / sigma2)
        grad_U += w / sigma2
        return U, grad_U

    def run_hmc(self,
                n_samples: int = 20,
                leapfrog_steps: int = 10,
                step_size: float = 0.05,
                adapt_step_size: bool = True) -> None:
        """
        Draw n_samples from p(w | data) using HMC with leapfrog integration.

        w ≥ 0 constraint handled by *reflection*: when a leapfrog step pushes
        w_c < 0, both position and momentum are negated for that dimension.
        This is a volume-preserving symplectic map — no Jacobian correction
        needed (Neal 2011 §5.4; Pakman & Paninski 2014).

        Step size is adapted toward 65% acceptance after each block.
        After updating weights, _lp_cache is cleared (log probs depend on weights).
        """
        if not self._train_pairs:
            return

        pair_data = self._build_pair_data()
        n_c = self.n_constraints
        current_w = self.weights.copy()
        eps = self.hmc_step_size
        n_accepted = 0

        for _ in range(n_samples):
            p = np.random.randn(n_c)
            current_U, _ = self._log_posterior_and_grad(current_w, pair_data)
            current_K = 0.5 * np.dot(p, p)

            proposed_w = current_w.copy()
            proposed_p = p.copy()

            _, grad_U = self._log_posterior_and_grad(proposed_w, pair_data)
            proposed_p -= 0.5 * eps * grad_U

            for _ in range(leapfrog_steps):
                proposed_w += eps * proposed_p
                neg_mask = proposed_w < 0.0          # reflect at w=0 boundary
                proposed_w[neg_mask] = -proposed_w[neg_mask]
                proposed_p[neg_mask] = -proposed_p[neg_mask]
                _, grad_U = self._log_posterior_and_grad(proposed_w, pair_data)
                proposed_p -= eps * grad_U

            _, grad_U = self._log_posterior_and_grad(proposed_w, pair_data)
            proposed_p -= 0.5 * eps * grad_U
            proposed_p = -proposed_p

            proposed_U, _ = self._log_posterior_and_grad(proposed_w, pair_data)
            proposed_K = 0.5 * np.dot(proposed_p, proposed_p)

            delta_H = (proposed_U + proposed_K) - (current_U + current_K)
            if random.random() < min(1.0, math.exp(-delta_H)):
                current_w = proposed_w.copy()
                n_accepted += 1

            self.weight_history.append(current_w.copy())

        self.weights = current_w.copy()
        self._lp_cache.clear()    # weights changed — invalidate log-prob cache
        self._wfst_dirty = True

        if adapt_step_size and n_samples > 0:
            self.hmc_step_size *= math.exp(0.1 * (n_accepted / n_samples - 0.65))
            self.hmc_step_size = float(np.clip(self.hmc_step_size, 1e-4, 2.0))

        self._hmc_accept_history.append(n_accepted / max(n_samples, 1))
        self._train_pairs.clear()

    def lexicon_ur(self, ur_counts: Dict[str, int]) -> Dict[str, str]:
        """
        For each UR, return the MAP SR under the current grammar.
        Uses cached candidates and log-probs — fast after warm-up.
        """
        result = {}
        for ur in ur_counts:
            cands = self._candidates_for(ur)
            result[ur] = max(cands, key=lambda s: self.log_prob(ur, s))
        return result


## 5. Display functions for constraint weights

Three functions for inspecting the grammar at any point:

- `display_initial_weights(grammar)` — shows p-map prior weights before sampling  
- `display_posterior_weights(grammar)` — shows posterior mean ± SD, HMC diagnostics, chain table, and ASCII trace plot  

**Reading the trace plot:** good mixing looks like a fuzzy horizontal band.
A trend (monotone drift) indicates the chain hasn't converged yet.
Acceptance rate ~0.65 is the HMC sweet spot; outside [0.3, 0.9] suggests tuning needed.


In [5]:
def display_initial_weights(grammar: MaxEntPhonology) -> None:
    """Print a formatted table of the p-map prior (initial) constraint weights."""
    _print_weight_table(
        grammar.constraints,
        grammar._initial_weights,
        title="Initial constraint weights (p-map prior)",
    )


def display_posterior_weights(grammar: MaxEntPhonology,
                               show_history: bool = True,
                               show_trace: bool = True) -> None:
    """
    Print posterior weight distribution from the HMC chain.

    Shows mean ± SD per constraint, HMC diagnostics (acceptance rate,
    adapted step size), optional chain table, and ASCII trace plot.

    Interpretation:
      FAITH-PMAP mean < prior  →  grammar learned cheap alternations (Zuraw 2013)
      Large SD                 →  weight uncertainty; more data needed
      Accept rate ~0.65        →  HMC well-tuned
      Accept rate < 0.3        →  reduce hmc_step_size
      Accept rate > 0.9        →  increase hmc_step_size (mixing too slow)
    """
    chain = grammar.weight_history
    n_c = grammar.n_constraints

    means = np.mean(chain, axis=0) if chain else grammar.weights.copy()
    sds   = np.std(chain,  axis=0) if chain else np.zeros(n_c)

    _print_weight_table(
        grammar.constraints, means,
        title=f"Posterior weight means  (chain length = {len(chain)})",
        reference=grammar._initial_weights,
        ref_label="Δ from prior",
        sds=sds,
    )

    if grammar._hmc_accept_history:
        mean_accept = float(np.mean(grammar._hmc_accept_history))
        print(f"  HMC diagnostics:")
        print(f"    Blocks run       : {len(grammar._hmc_accept_history)}")
        print(f"    Mean accept rate : {mean_accept:.3f}  (target ~0.65)")
        print(f"    Adapted step ε   : {grammar.hmc_step_size:.5f}")
        if mean_accept < 0.3:
            print("    ⚠  Low acceptance — consider reducing hmc_step_size")
        elif mean_accept > 0.9:
            print("    ⚠  High acceptance — sampler may be moving slowly")
        print()

    if show_history and chain:
        col_w = 10
        names = [c.name for c in grammar.constraints]
        header = f"  {'Sample':<8}" + "".join(f"  {n:<{col_w}}" for n in names)
        sep = "  " + "-" * (len(header) - 2)
        step = max(1, len(chain) // 40)
        shown = chain[::step]
        print(f"  HMC chain (every {step} sample(s), {len(shown)} rows shown):")
        print(sep); print(header); print(sep)
        print(f"  {'prior':<8}" + "".join(f"  {w:<{col_w}.4f}" for w in grammar._initial_weights))
        for si, wv in enumerate(shown, start=1):
            print(f"  {si*step:<8}" + "".join(f"  {w:<{col_w}.4f}" for w in wv))
        print(sep)

    if show_trace and chain and n_c > 0:
        _ascii_trace(
            [float(w[0]) for w in chain],
            label=grammar.constraints[0].name,
            prior=float(grammar._initial_weights[0]),
        )


def _ascii_trace(samples: List[float], label: str, prior: float,
                 width: int = 60, height: int = 8) -> None:
    """ASCII trace plot — useful for diagnosing convergence and mixing."""
    if not samples:
        return
    lo, hi = min(samples), max(samples)
    if hi == lo:
        hi = lo + 0.01
    print(f"\n  Trace plot: {label}")
    print(f"  {hi:.3f} ┤")
    for row_i in range(height):
        threshold = hi - (row_i + 0.5) * (hi - lo) / height
        step = max(1, len(samples) // width)
        line = "".join("█" if samples[col_i * step] >= threshold else " "
                       for col_i in range(min(width, len(samples))))
        y_label = f"{hi - row_i*(hi-lo)/height:.3f}" if row_i % 3 == 0 else "      "
        print(f"  {y_label:>7} │{line}│")
    print(f"  {lo:.3f} ┤")
    prior_row = int(round((hi - prior) / (hi - lo) * height))
    print(f"  {'prior→':>7}  " + " " * min(prior_row * 2, width) + "↑")
    print(f"  {'':>7}  0" + " " * (width // 2 - 4) + f"iter {len(samples)}")
    print()


def _print_weight_table(
        constraints: List[Constraint],
        weights: np.ndarray,
        title: str,
        reference: Optional[np.ndarray] = None,
        ref_label: str = "",
        sds: Optional[np.ndarray] = None,
        bar_width: int = 20) -> None:
    """Formatted table helper. Shows mean±SD column if sds provided, Δ column if reference provided."""
    max_w = max(float(weights.max()), 1e-3)
    name_w   = max(len(c.name) for c in constraints) + 2
    show_sd  = sds is not None
    show_ref = reference is not None
    w_col    = 18 if show_sd else 12
    ref_col  = 14 if show_ref else 0
    border_len = name_w + w_col + bar_width + 4 + ref_col
    border = "─" * border_len
    print(); print(f"  ┌{border}┐")
    print(f"  │  {title:<{border_len-2}}│")
    row_sep = (f"  ├{'─'*name_w}┬{'─'*w_col}┬{'─'*(bar_width+2)}"
               + (f"┬{'─'*ref_col}" if show_ref else "") + "┤")
    print(row_sep)
    w_header = "Mean ± SD" if show_sd else "Weight"
    print(f"  │  {'Constraint':<{name_w-2}}│  {w_header:<{w_col-2}}│  {'Bar':<{bar_width}}"
          + (f"│  {ref_label:<{ref_col-2}}" if show_ref else "") + "│")
    print(row_sep)
    for i, c in enumerate(constraints):
        w = float(weights[i])
        w_str = (f"{w:.3f} ± {float(sds[i]):.3f}" if show_sd else f"{w:.4f}")
        bar = "█" * int(round(bar_width * w / max_w)) + "░" * (bar_width - int(round(bar_width * w / max_w)))
        delta_col = ""
        if show_ref:
            delta = w - float(reference[i])
            delta_col = f"│  {'+' if delta>=0 else ''}{delta:<{ref_col-4}.4f}"
        print(f"  │  {c.name:<{name_w-2}}│  {w_str:<{w_col-2}}│  {bar}" + delta_col + "│")
    print(f"  └{'─'*name_w}┴{'─'*w_col}┴{'─'*(bar_width+2)}"
          + (f"┴{'─'*ref_col}" if show_ref else "") + "┘")
    print()


## 6. Pitman-Yor process cache

The PY cache implements the Chinese Restaurant Process for URs:

$$P(u_{n+1} = w \mid \text{cache}) \propto \begin{cases} n_w - d \cdot t_w & \text{(sit at existing table for } w\text{)} \\ (\alpha + d \cdot T) \cdot G_0(w) & \text{(open new table)} \end{cases}$$

where $n_w$ = token count, $t_w$ = table count, $T$ = total tables, $d$ = discount, $\alpha$ = concentration.

- Setting $d > 0$ gives a **power-law** frequency distribution (Zipf), matching natural lexicons.  
- $d = 0$ recovers the Dirichlet Process (Goldwater et al. 2009).


In [6]:
class PitmanYorCache:
    """
    Memoisation table implementing the Pitman-Yor process.
    The PY cache is placed on underlying representations (URs), giving the
    model an emergent lexicon with power-law frequency distribution.

    _lexicon_cache is invalidated on every add() or remove() and rebuilt
    lazily on the next lexicon() call, avoiding O(|vocab|) dict comprehensions
    on every span scoring call.
    """
    def __init__(self, discount: float = 0.5, concentration: float = 1.0):
        assert 0.0 <= discount < 1.0
        assert concentration > 0.0
        self.d = discount
        self.alpha = concentration
        self.table_counts: Dict[str, int] = defaultdict(int)
        self.customer_counts: Dict[str, int] = defaultdict(int)
        self.total_tables = 0
        self.total_customers = 0
        self._lexicon_cache: Optional[Dict[str, int]] = None   # lazy cache

    def predictive_score(self, word: str, base_prob: float) -> float:
        n = self.customer_counts[word]
        t = self.table_counts[word]
        denom = self.total_customers + self.alpha + 1e-300
        return max((n - self.d * t) / denom
                   + (self.alpha + self.d * self.total_tables) / denom * base_prob,
                   1e-300)

    def add(self, word: str, base_prob: float) -> None:
        self._lexicon_cache = None    # invalidate
        n = self.customer_counts[word]
        denom = n + (self.alpha + self.d * self.total_tables) * base_prob + 1e-300
        if random.random() < (self.alpha + self.d * self.total_tables) * base_prob / denom:
            self.table_counts[word] += 1
            self.total_tables += 1
        self.customer_counts[word] += 1
        self.total_customers += 1

    def remove(self, word: str) -> None:
        self._lexicon_cache = None    # invalidate
        assert self.customer_counts[word] > 0
        t, n = self.table_counts[word], self.customer_counts[word]
        if t > 0 and random.random() < min(max((t * self.d) / (n + 1e-9), 0.0), 1.0):
            self.table_counts[word] -= 1
            self.total_tables -= 1
        self.customer_counts[word] -= 1
        self.total_customers -= 1

    def lexicon(self) -> Dict[str, int]:
        """Return {ur: count} for all active URs. Result is cached until next add/remove."""
        if self._lexicon_cache is None:
            self._lexicon_cache = {w: c for w, c in self.customer_counts.items() if c > 0}
        return self._lexicon_cache


## 7. Character base distribution (G₀) and UR proposal distribution

**G₀** is the base distribution from which novel URs are generated:  
geometric word length × empirical character unigram.

**URProposer** generates candidate URs for each observed SR during sampling.
Proposals are p-map-weighted: edits with lower perceptual cost get higher
proposal weight, making the importance sampler efficient for common alternations.


In [7]:
class CharacterBaseDistribution:
    """G₀: geometric length × character unigram model for novel URs."""

    def __init__(self, alphabet: str, p_end: float = 0.5, char_alpha: float = 1.0):
        self.alphabet = list(alphabet)
        self.V = len(self.alphabet)
        self.p_end = p_end
        self.char_alpha = char_alpha
        self.char_counts: Dict[str, float] = defaultdict(float)
        self.total_chars: float = 0.0

    def char_prob(self, c: str) -> float:
        return (self.char_counts[c] + self.char_alpha) / (
            self.total_chars + self.char_alpha * self.V)

    def word_prob(self, word: str) -> float:
        if not word:
            return 0.0
        prob = (1.0 - self.p_end) ** (len(word) - 1) * self.p_end
        for c in word:
            prob *= self.char_prob(c)
        return prob

    def update_counts(self, word: str, delta: float = 1.0) -> None:
        for c in word:
            self.char_counts[c] += delta
            self.total_chars += delta


class URProposer:
    """
    Generate candidate URs for a given surface form.

    Proposal sources (in order of priority):
      1. The SR itself (faithful candidate, strong prior weight).
      2. All URs already in the PY lexicon (rich-get-richer reuse).
      3. Single-operation p-map-weighted edits of the SR.
         Proposal weight ∝ 1 / (pmap_distance + 0.5) so that perceptually
         cheap alternations like d→ð are proposed more frequently.
    """

    def __init__(self, alphabet: str, n_edit_proposals: int = 15):
        self.alphabet = list(alphabet)

    def propose(self, sr: str, lexicon: Dict[str, int]) -> List[Tuple[str, float]]:
        proposals: Dict[str, float] = {sr: 5.0}
        for ur_type, cnt in lexicon.items():
            if abs(len(ur_type) - len(sr)) <= 2:
                proposals[ur_type] = proposals.get(ur_type, 0.0) + cnt
        for i in range(len(sr)):
            for c in self.alphabet:
                if c != sr[i]:
                    candidate = sr[:i] + c + sr[i+1:]
                    proposals[candidate] = proposals.get(candidate, 0.0) + 1.0 / (pmap_distance(sr[i], c) + 0.5)
            candidate = sr[:i] + sr[i+1:]
            if candidate:
                proposals[candidate] = proposals.get(candidate, 0.0) + 0.5
        for i in range(len(sr) + 1):
            for c in self.alphabet:
                candidate = sr[:i] + c + sr[i:]
                if len(candidate) <= len(sr) + 1:
                    proposals[candidate] = proposals.get(candidate, 0.0) + 0.2
        total = sum(proposals.values())
        return [(ur, w / total) for ur, w in proposals.items()]


## 8. Main sampler: `PhonologicalAdaptorSegmenter`

This ties everything together. The generative model is:

1. **Segmentation** — draw a partition of the utterance into $n$ surface word tokens $s_1 \ldots s_n$  
2. **UR assignment** — for each token $s_i$, draw $u_i \sim \text{PY}(u_i \mid \text{cache}, G_0)$  
3. **Channel** — observe $s_i \sim \text{MaxEnt}(u_i, \mathbf{w})$  

Inference is **interleaved Gibbs + HMC**:

```
for each outer Gibbs sweep:
    for each utterance:
        remove its (UR, SR) pairs from PY cache
        forward-DP over boundaries → marginalise over URs
        backward-sample new boundaries + URs
        re-add to PY cache; buffer (UR, SR) pairs
    if sweep % hmc_every == 0:
        run_hmc(n_samples)  →  draw fresh w from p(w | buffered pairs)
```


In [8]:
class PhonologicalAdaptorSegmenter:
    """
    Full Bayesian segmenter: PY cache over URs + MaxEnt channel + HMC weights.

    Parameters
    ----------
    utterances          : list of unsegmented phonemic strings
    py_discount         : PY discount d ∈ [0,1)  (0 → DP; >0 → power-law)
    py_concentration    : PY concentration α > 0
    p_end               : geometric word-length parameter
    pmap_init_weight    : initial FAITH-PMAP weight (p-map prior)
    hmc_every           : run HMC after every this many Gibbs sweeps
    hmc_samples         : HMC samples per block
    hmc_leapfrog_steps  : leapfrog integration steps per HMC proposal
    hmc_step_size       : initial leapfrog step size ε (adapted automatically)
    max_word_len        : maximum surface word length considered
    n_ur_proposals      : number of UR edit proposals per surface token
    """

    def __init__(self,
                 utterances: List[str],
                 alphabet: Optional[str] = None,
                 py_discount: float = 0.5,
                 py_concentration: float = 1.0,
                 p_end: float = 0.4,
                 pmap_init_weight: float = 5.0,
                 hmc_every: int = 10,
                 hmc_samples: int = 20,
                 hmc_leapfrog_steps: int = 10,
                 hmc_step_size: float = 0.05,
                 max_word_len: int = 12,
                 n_ur_proposals: int = 20):
        self.utterances = utterances
        self.alphabet = alphabet or _infer_alphabet(utterances)
        self.max_word_len = max_word_len
        self.hmc_every = hmc_every
        self.hmc_samples = hmc_samples
        self.hmc_leapfrog_steps = hmc_leapfrog_steps

        self.cache    = PitmanYorCache(py_discount, py_concentration)
        self.base     = CharacterBaseDistribution(self.alphabet, p_end)
        self.grammar  = MaxEntPhonology(self.alphabet,
                                        pmap_init_weight=pmap_init_weight,
                                        hmc_step_size=hmc_step_size,
                                        hmc_leapfrog_steps=hmc_leapfrog_steps)
        self.proposer = URProposer(self.alphabet, n_edit_proposals=n_ur_proposals)

        self.states: List[List[Tuple[str, str]]] = [[(u, u)] for u in utterances]
        for seg_list in self.states:
            for sr, ur in seg_list:
                self.cache.add(ur, self.base.word_prob(ur))
                self.base.update_counts(ur, delta=1.0)

        self._ur_posterior: Dict[str, float] = defaultdict(float)

    def _joint_score(self, sr: str, ur: str) -> float:
        base_p = self.base.word_prob(ur)
        return (math.log(self.cache.predictive_score(ur, base_p) + 1e-300)
                + self.grammar.log_prob(ur, sr))

    def _sample_ur(self, sr: str) -> Tuple[str, float]:
        proposals = self.proposer.propose(sr, self.cache.lexicon())
        log_scores, candidates = [], []
        for (ur_cand, q_weight) in proposals:
            base_p = self.base.word_prob(ur_cand)
            log_target = (math.log(self.cache.predictive_score(ur_cand, base_p) + 1e-300)
                          + self.grammar.log_prob(ur_cand, sr)
                          - math.log(q_weight + 1e-300))
            log_scores.append(log_target)
            candidates.append(ur_cand)
        arr = np.array(log_scores); arr -= arr.max()
        probs = np.exp(arr); probs /= probs.sum()
        idx = int(np.random.choice(len(candidates), p=probs))
        return candidates[idx], float(probs[idx])

    def _sample_segmentation(self, surface: str) -> List[Tuple[str, str]]:
        """Forward–backward DP over boundary positions, marginalising over URs."""
        n = len(surface)
        NEG_INF = -1e30
        alpha = [NEG_INF] * (n + 1)
        alpha[0] = 0.0
        span_cache: Dict[Tuple[int,int], Tuple[float, str]] = {}

        for j in range(1, n + 1):
            for i in range(max(0, j - self.max_word_len), j):
                if alpha[i] == NEG_INF:
                    continue
                sr_span = surface[i:j]
                if (i, j) not in span_cache:
                    proposals = self.proposer.propose(sr_span, self.cache.lexicon())
                    ur_scores = [(self._joint_score(sr_span, ur_c), ur_c)
                                 for (ur_c, _) in proposals[:10]]
                    log_marg = _logsumexp([lp for lp, _ in ur_scores])
                    span_cache[(i, j)] = (log_marg, max(ur_scores, key=lambda x: x[0])[1])
                new_score = alpha[i] + span_cache[(i, j)][0]
                if new_score > alpha[j]:
                    alpha[j] = new_score

        result: List[Tuple[str, str]] = []
        pos = n
        while pos > 0:
            opts = [(i, pos, alpha[i] + span_cache[(i, pos)][0])
                    for i in range(max(0, pos - self.max_word_len), pos)
                    if alpha[i] != NEG_INF and (i, pos) in span_cache]
            if not opts:
                break
            scores = np.array([sc for _, _, sc in opts]); scores -= scores.max()
            probs = np.exp(scores); probs /= probs.sum()
            i, j, _ = opts[int(np.random.choice(len(opts), p=probs))]
            ur_sampled, _ = self._sample_ur(surface[i:j])
            result.append((surface[i:j], ur_sampled))
            pos = i

        result.reverse()
        return result if result else [(surface, surface)]

    def sweep(self, sweep_idx: int = 0) -> None:
        for utt_idx, (utt, seg_list) in enumerate(zip(self.utterances, self.states)):
            for sr, ur in seg_list:
                self.cache.remove(ur)
                self.base.update_counts(ur, delta=-1.0)
            new_seg = self._sample_segmentation(utt)
            for sr, ur in new_seg:
                self.cache.add(ur, self.base.word_prob(ur))
                self.base.update_counts(ur, delta=1.0)
                self.grammar.accumulate(ur, sr)
            self.states[utt_idx] = new_seg
        if (sweep_idx + 1) % self.hmc_every == 0:
            self.grammar.run_hmc(n_samples=self.hmc_samples,
                                 leapfrog_steps=self.hmc_leapfrog_steps,
                                 step_size=self.grammar.hmc_step_size,
                                 adapt_step_size=True)

    def run(self, n_iter: int = 300, burn_in: int = 150,
            print_every: int = 50) -> Dict:
        """
        Run n_iter Gibbs sweeps. After burn_in, accumulate posterior samples.

        Returns
        -------
        dict with keys:
          'ur_lexicon'         {ur: posterior_freq}
          'ur_to_sr'           {ur: MAP_sr}  under posterior mean grammar
          'segmentations'      final List[List[(sr, ur)]] per utterance
          'weight_posterior'   {constraint: {'mean', 'sd', 'samples'}}
          'hmc_acceptance_rate' float
          'hmc_step_size'      float (final adapted value)
        """
        n_hmc_before_burnin = burn_in // self.hmc_every
        for it in range(n_iter):
            self.sweep(sweep_idx=it)
            if it >= burn_in:
                for ur, cnt in self.cache.lexicon().items():
                    self._ur_posterior[ur] += cnt
            if (it + 1) % print_every == 0:
                chain_len = len(self.grammar.weight_history)
                accept_str = (f"  HMC accept={self.grammar._hmc_accept_history[-1]:.2f}"
                              if self.grammar._hmc_accept_history else "")
                w_str = ', '.join(f"{c.name}={w:.2f}"
                                  for c, w in zip(self.grammar.constraints, self.grammar.weights))
                print(f"Iter {it+1:4d} | {len(self.cache.lexicon()):3d} UR types | "
                      f"w=[{w_str}] | chain={chain_len}{accept_str}")

        total = sum(self._ur_posterior.values()) or 1.0
        ur_lexicon = {ur: cnt/total for ur, cnt in
                      sorted(self._ur_posterior.items(), key=lambda x: -x[1])}

        post_samples = self.grammar.weight_history[n_hmc_before_burnin * self.hmc_samples:]
        mean_w = np.mean(post_samples, axis=0) if post_samples else self.grammar.weights.copy()
        saved = self.grammar.weights.copy()
        self.grammar.weights = mean_w; self.grammar._wfst_dirty = True
        ur_to_sr = self.grammar.lexicon_ur({ur: c for ur, c in self._ur_posterior.items()})
        self.grammar.weights = saved; self.grammar._wfst_dirty = True

        weight_posterior = {}
        for ci, constraint in enumerate(self.grammar.constraints):
            samples = [float(w[ci]) for w in post_samples] if post_samples else []
            weight_posterior[constraint.name] = {
                'mean': float(np.mean(samples)) if samples else float(self.grammar.weights[ci]),
                'sd':   float(np.std(samples))  if samples else 0.0,
                'samples': samples,
            }
        hmc_accept = (float(np.mean(self.grammar._hmc_accept_history))
                      if self.grammar._hmc_accept_history else float('nan'))
        return {
            'ur_lexicon': ur_lexicon, 'ur_to_sr': ur_to_sr,
            'segmentations': self.states,
            'weight_posterior': weight_posterior,
            'hmc_acceptance_rate': hmc_accept,
            'hmc_step_size': self.grammar.hmc_step_size,
        }


## 9. Utility functions

Internal helpers: `_infer_alphabet`, `_logsumexp`, `_levenshtein_alignment`, `_random_edit`.  
These are used throughout the model but not part of the public API.

> ⚠️ **Run this cell before running the demo.** The utility functions (`_logsumexp`, `_levenshtein_alignment`, etc.) are called inside `MaxEntPhonology` and `PhonologicalAdaptorSegmenter` methods. Python resolves them at call-time, not definition-time, so the notebook will work fine if run top-to-bottom — but if you jump straight to the demo, run this cell first.


In [9]:
def _infer_alphabet(utterances: List[str]) -> str:
    return ''.join(sorted(set(''.join(utterances))))


def _logsumexp(vals: List[float]) -> float:
    if not vals:
        return -1e30
    m = max(vals)
    return m + math.log(sum(math.exp(v - m) for v in vals) + 1e-300)


def _logsumexp_np(arr: np.ndarray) -> float:
    if len(arr) == 0:
        return -1e30
    m = arr.max()
    return float(m + np.log(np.sum(np.exp(arr - m)) + 1e-300))


def _levenshtein_alignment(s: str, t: str) -> List[Tuple[Optional[str], Optional[str]]]:
    """Levenshtein-optimal character alignment used by constraint violation functions."""
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = dp[i-1][j-1] if s[i-1]==t[j-1] else 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    alignment, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and s[i-1] == t[j-1]:
            alignment.append((s[i-1], t[j-1])); i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            alignment.append((s[i-1], t[j-1])); i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            alignment.append((s[i-1], None)); i -= 1
        else:
            alignment.append((None, t[j-1])); j -= 1
    alignment.reverse()
    return alignment


def _random_edit(word: str, alphabet: List[str]) -> str:
    """Apply one random edit operation (sub / del / ins) to word."""
    if not word:
        return random.choice(alphabet)
    op = random.choice(['sub', 'del', 'ins'])
    i = random.randint(0, len(word) - 1)
    if op == 'sub':   return word[:i] + random.choice(alphabet) + word[i+1:]
    elif op == 'del': return word[:i] + word[i+1:] or word
    else:             return word[:i] + random.choice(alphabet) + word[i:]


## 10. Demo: toy phonemic corpus with th-stopping

The toy corpus simulates child-directed English transcribed at the phoneme level.
Key phonological process: **th-stopping** — /ð/ is frequently realised as [d]
(common in NYC English, L1/L2 acquisition, and informal speech).

The model should:
1. Correctly segment the utterances into words (e.g. `juwAnt | da | dog`)
2. Learn URs like `/ðə/` for the surface form `[da]`
3. **Reduce** the FAITH-PMAP weight relative to the prior, because d↔ð is a
   perceptually cheap alternation that appears frequently in the data

> **Note:** This is a minimal corpus for illustration. On real data (e.g. the
> Brent & Cartwright 1996 corpus in `brent-phone.tar.gz`) the model will need
> 100–500 iterations and more HMC samples to converge well.

### ⚙️ Configurable parameters


In [13]:
# ── Corpus ────────────────────────────────────────────────────────────────
# ASCII phoneme encoding:  D = [ð],  S = [ʃ],  N = [ŋ],  @ = schwa
utterances = [
    "juwAndusiydAbook",
    "juwAnthAvdAbook",
    "Arjugoingdusiydadog",
    "dAdAwAnttusiydadog",
    "juwAndusiydakAt",
    "siydakAt",
    "siydAbook",
    "siydadog",
    "juwAndakAt",
    "juwAndadog",
    "lukAtdadog",
    "lukAtdakAt",
]

# ── Hyperparameters ───────────────────────────────────────────────────────
PY_DISCOUNT         = 0.5     # d: 0 → DP; >0 → power-law lexicon
PY_CONCENTRATION    = 2.0     # α: higher → more word types
P_END               = 0.3     # geometric word-length stop probability
PMAP_INIT_WEIGHT    = 4.0     # initial FAITH-PMAP weight (p-map prior)

HMC_EVERY           = 10      # run HMC every this many Gibbs sweeps
HMC_SAMPLES         = 20      # HMC samples per block
HMC_LEAPFROG_STEPS  = 10      # leapfrog integration steps per proposal
HMC_STEP_SIZE       = 0.05    # initial ε (adapted toward 65% acceptance)

MAX_WORD_LEN        = 8       # max surface word length to consider
N_ITER              = 150     # total Gibbs sweeps
BURN_IN             = 75      # discard first BURN_IN sweeps from posteriors
PRINT_EVERY         = 1      # progress reporting interval

print(f"Corpus:  {len(utterances)} utterances")
print(f"Backend: {'pynini WFST (exact Z)' if PYNINI_AVAILABLE else 'Python DP (len±2 approx)'}")


Corpus:  12 utterances
Backend: pynini WFST (exact Z)


### Run the sampler


In [14]:
# Instantiate and show prior weights
seg = PhonologicalAdaptorSegmenter(
    utterances,
    py_discount=PY_DISCOUNT,
    py_concentration=PY_CONCENTRATION,
    p_end=P_END,
    pmap_init_weight=PMAP_INIT_WEIGHT,
    hmc_every=HMC_EVERY,
    hmc_samples=HMC_SAMPLES,
    hmc_leapfrog_steps=HMC_LEAPFROG_STEPS,
    hmc_step_size=HMC_STEP_SIZE,
    max_word_len=MAX_WORD_LEN,
)

display_initial_weights(seg.grammar)



  ┌────────────────────────────────────────────────┐
  │  Initial constraint weights (p-map prior)      │
  ├────────────┬────────────┬──────────────────────┤
  │  Constraint│  Weight    │  Bar                 │
  ├────────────┬────────────┬──────────────────────┤
  │  FAITH-PMAP│  4.0000    │  ████████████████████│
  │  MAX       │  0.1000    │  ░░░░░░░░░░░░░░░░░░░░│
  │  DEP       │  0.1000    │  ░░░░░░░░░░░░░░░░░░░░│
  └────────────┴────────────┴──────────────────────┘



In [15]:
# Run the sampler
results = seg.run(n_iter=N_ITER, burn_in=BURN_IN, print_every=PRINT_EVERY)


Iter    1 |  17 UR types | w=[FAITH-PMAP=4.00, MAX=0.10, DEP=0.10] | chain=0
Iter    2 |  16 UR types | w=[FAITH-PMAP=4.00, MAX=0.10, DEP=0.10] | chain=0
Iter    3 |  17 UR types | w=[FAITH-PMAP=4.00, MAX=0.10, DEP=0.10] | chain=0
Iter    4 |  15 UR types | w=[FAITH-PMAP=4.00, MAX=0.10, DEP=0.10] | chain=0
Iter    5 |  15 UR types | w=[FAITH-PMAP=4.00, MAX=0.10, DEP=0.10] | chain=0
Iter    6 |  13 UR types | w=[FAITH-PMAP=4.00, MAX=0.10, DEP=0.10] | chain=0
Iter    7 |  13 UR types | w=[FAITH-PMAP=4.00, MAX=0.10, DEP=0.10] | chain=0
Iter    8 |  12 UR types | w=[FAITH-PMAP=4.00, MAX=0.10, DEP=0.10] | chain=0
Iter    9 |  13 UR types | w=[FAITH-PMAP=4.00, MAX=0.10, DEP=0.10] | chain=0
Iter   10 |  12 UR types | w=[FAITH-PMAP=4.00, MAX=0.10, DEP=0.10] | chain=20  HMC accept=0.00
Iter   11 |  14 UR types | w=[FAITH-PMAP=4.00, MAX=0.10, DEP=0.10] | chain=20  HMC accept=0.00
Iter   12 |  15 UR types | w=[FAITH-PMAP=4.00, MAX=0.10, DEP=0.10] | chain=20  HMC accept=0.00
Iter   13 |  12 UR typ

In [12]:
import cProfile, pstats, io

pr = cProfile.Profile()
pr.enable()
seg.run(n_iter=5, burn_in=0, print_every=5)
pr.disable()

s = io.StringIO()
ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
ps.print_stats(20)
print(s.getvalue())

Iter    5 |  13 UR types | w=[FAITH-PMAP=4.00, MAX=0.10, DEP=0.10] | chain=0
         511158196 function calls (511158100 primitive calls) in 320.054 seconds

   Ordered by: cumulative time
   List reduced from 309 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000  320.053  160.027 c:\Users\canaa\miniconda3\envs\phon-adaptor\Lib\site-packages\IPython\core\interactiveshell.py:3665(run_code)
      4/2    0.000    0.000  320.053  160.027 {built-in method builtins.exec}
        1    0.000    0.000  320.053  320.053 C:\Users\canaa\AppData\Local\Temp\ipykernel_37460\55353655.py:134(run)
    79840    0.172    0.000  318.021    0.004 C:\Users\canaa\AppData\Local\Temp\ipykernel_37460\360137895.py:98(log_prob)
    20335    0.450    0.000  317.848    0.016 C:\Users\canaa\AppData\Local\Temp\ipykernel_37460\360137895.py:108(_log_prob_dp)
        5    0.002    0.000  317.525   63.505 C:\Users\canaa\AppData\Local\Temp\ip

### Inspect results


In [16]:
# Posterior constraint weights + HMC diagnostics
display_posterior_weights(seg.grammar, show_history=True, show_trace=True)



  ┌────────────────────────────────────────────────────────────────────┐
  │  Posterior weight means  (chain length = 300)                      │
  ├────────────┬──────────────────┬──────────────────────┬──────────────┤
  │  Constraint│  Mean ± SD       │  Bar                 │  Δ from prior│
  ├────────────┬──────────────────┬──────────────────────┬──────────────┤
  │  FAITH-PMAP│  1.784 ± 1.562   │  ████████████████████│  -2.2156   │
  │  MAX       │  0.075 ± 0.036   │  █░░░░░░░░░░░░░░░░░░░│  -0.0250   │
  │  DEP       │  0.602 ± 0.359   │  ███████░░░░░░░░░░░░░│  +0.5018    │
  └────────────┴──────────────────┴──────────────────────┴──────────────┘

  HMC diagnostics:
    Blocks run       : 15
    Mean accept rate : 0.057  (target ~0.65)
    Adapted step ε   : 0.02053
    ⚠  Low acceptance — consider reducing hmc_step_size

  HMC chain (every 7 sample(s), 43 rows shown):
  --------------------------------------------
  Sample    FAITH-PMAP  MAX         DEP       
  -----------------

In [17]:
# Weight posterior summary
print("--- Weight posterior summary ---")
for cname, stats in results['weight_posterior'].items():
    print(f"  {cname:<20s}  mean={stats['mean']:.4f}  sd={stats['sd']:.4f}")
print(f"\n  HMC acceptance rate : {results['hmc_acceptance_rate']:.3f}  (target ~0.65)")
print(f"  Adapted step size ε : {results['hmc_step_size']:.5f}")


--- Weight posterior summary ---
  FAITH-PMAP            mean=0.6175  sd=0.4104
  MAX                   mean=0.0713  sd=0.0252
  DEP                   mean=0.8092  sd=0.1518

  HMC acceptance rate : 0.057  (target ~0.65)
  Adapted step size ε : 0.02053


In [18]:
# Posterior UR lexicon with MAP surface forms
print("--- Posterior UR Lexicon (top 20, with MAP surface form) ---")
print(f"  {'UR':<18} {'MAP SR':<18} {'Post. freq':>10}  note")
print("  " + "-" * 55)
for ur, freq in list(results['ur_lexicon'].items())[:20]:
    sr_map = results['ur_to_sr'].get(ur, ur)
    note = "← SR differs from UR" if sr_map != ur else ""
    print(f"  {ur:<18} {sr_map:<18} {freq:>10.4f}  {note}")


--- Posterior UR Lexicon (top 20, with MAP surface form) ---
  UR                 MAP SR             Post. freq  note
  -------------------------------------------------------
  siydadog           siydadog               0.3809  
  juwAnd             juwAnd                 0.2577  
  d                  d                      0.0675  
  u                  u                      0.0328  
  o                  o                      0.0323  
  g                  g                      0.0298  
  i                  i                      0.0244  
  a                  a                      0.0186  
  s                  s                      0.0152  
  A                  A                      0.0127  
  n                  n                      0.0093  
  y                  y                      0.0093  
  w                  w                      0.0073  
  j                  j                      0.0073  
  dd                 dd                     0.0044  
  odog               odog    

In [ ]:
# Final segmentations
print("--- Final Segmentations (surface_word→UR  |  ...) ---")
for utt, seg_list in zip(utterances, results['segmentations']):
    seg_str = ' | '.join(f"{sr}→{ur}" if sr != ur else sr for sr, ur in seg_list)
    print(f"  Input : {utt}")
    print(f"  Output: {seg_str}")
    print()


## 11. Extension points

### Adding a custom markedness constraint

Subclass `Constraint` and pass it to `MaxEntPhonology`:

```python
class NoCodaObstruent(Constraint):
    """*CODA-OBS: penalise voiced obstruents in coda position."""
    def __init__(self):
        self.name = 'NO-CODA-OBS'

    def violations(self, ur: str, sr: str) -> float:
        # For each SR segment, check if it's a voiced obstruent in coda position.
        # 'Coda' approximation: any segment followed by a consonant or end-of-word.
        v = 0.0
        for i, c in enumerate(sr):
            is_final = (i == len(sr) - 1)
            next_is_C = (not is_final) and MANNER.get(sr[i+1], 'other') != 'vowel'
            if (is_final or next_is_C) and VOICING.get(c, '-') == '+' and MANNER.get(c, 'other') != 'vowel':
                v += 1.0
        return v

# Use custom constraints:
seg = PhonologicalAdaptorSegmenter(
    utterances,
    # ... other params ...
)
# Override grammar after init:
seg.grammar.constraints = _default_constraint_set() + [NoCodaObstruent()]
seg.grammar.n_constraints = len(seg.grammar.constraints)
```

### Plotting the weight posterior with matplotlib

```python
import matplotlib.pyplot as plt

wp = results['weight_posterior']
fig, axes = plt.subplots(1, len(wp), figsize=(4 * len(wp), 3))
for ax, (cname, stats) in zip(axes, wp.items()):
    ax.hist(stats['samples'], bins=20, color='steelblue', edgecolor='white')
    ax.axvline(stats['mean'], color='firebrick', linewidth=2, label=f"mean={stats['mean']:.2f}")
    ax.axvline(seg.grammar._initial_weights[list(wp.keys()).index(cname)],
               color='orange', linestyle='--', label='prior')
    ax.set_title(cname); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
```

### Using a real corpus

Download the Brent & Cartwright (1996) corpus from the PyAdaGram repo:
```bash
# In terminal:
wget https://github.com/kzhai/PyAdaGram/raw/master/brent-phone.tar.gz
tar xzf brent-phone.tar.gz
```
Then load utterances with:
```python
with open('brent-phone/br-phon.txt') as f:
    utterances = [line.strip() for line in f if line.strip()]
```
